用于处理银行线下excel数据,并将结果导入ES库
tree_search建立索引的数据来源获取
```
pip install elasticsearch==8.19.2
```

In [ ]:
import pandas as pd
data = pd.read_csv("../data/河南省证券交易所数据集.csv", encoding='gbk')
print("data:",len(data))

data: 100


In [12]:
data.head()

,证券交易所,历史检索热度
0,中原证券交易中心,57
1,黄河金融交易所,34
2,郑州国际证券交易平台,69
3,豫州资本交易所,85
4,嵩山股权交易中心,34


In [13]:
new_headers = {'证券交易所': 'finance_name', '历史检索热度': 'hot_freq'}
data.rename(columns=new_headers, inplace=True)
data.head()

,finance_name,hot_freq
0,中原证券交易中心,57
1,黄河金融交易所,34
2,郑州国际证券交易平台,69
3,豫州资本交易所,85
4,嵩山股权交易中心,34


In [ ]:
from elasticsearch import Elasticsearch
import warnings
from rag.rag_lecture_materials.config.config import RagConfig
warnings.filterwarnings("ignore")

In [ ]:
es_host = RagConfig().elastic_host
es_port = 9200
es = Elasticsearch(
    hosts=[{"host": es_host, "port": es_port, "scheme": "https"}],
    basic_auth=(RagConfig().elastic_username, RagConfig().elastic_password), #
    verify_certs=False)
es.info()

ObjectApiResponse({'name': '92ac0d2b5265', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'xB_7dTCGQ4-20-CPA1jSuA', 'version': {'number': '8.19.18', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'e8ac685d1710aae2c9fc9ca61e2956ab9424d5f8', 'build_date': '2026-06-26T10:09:47.981719133Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [35]:
# 删除索引
if es.indices.exists(index="finance_search_index"):
    es.indices.delete(index="finance_search_index")

In [7]:
# 创建ES bank_name_search_index索引
mappings = {
    "mappings": {
        "properties": {
            "finance_name": {
                "type": "text"  #金融机构名称，类型为text的会调用IK分词器进行分词
            },
            # "finance_name": {
            #     "type": "keyword"  # 不会分词
            # },
            "hot_freq":{
                "type": "int"
            }
        }
    }
}
# 创建索引
response = es.indices.create(index="finance_search_index", body=mappings, ignore=400)
response

ObjectApiResponse({'error': {'root_cause': [{'type': 'mapper_parsing_exception', 'reason': 'The mapper type [int] declared on field [hot_freq] does not exist. It might have been created within a future version or requires a plugin to be installed. Check the documentation.'}], 'type': 'mapper_parsing_exception', 'reason': 'Failed to parse mapping: The mapper type [int] declared on field [hot_freq] does not exist. It might have been created within a future version or requires a plugin to be installed. Check the documentation.', 'caused_by': {'type': 'mapper_parsing_exception', 'reason': 'The mapper type [int] declared on field [hot_freq] does not exist. It might have been created within a future version or requires a plugin to be installed. Check the documentation.'}}, 'status': 400})

In [37]:
# 逐行处理金融机构数据写入ES
import time
start_time=time.time()
for i,(finance_name, hot_freq) in enumerate(zip(data['finance_name'], data['hot_freq'])):
    body = {"finance_name": finance_name,"hot_freq":hot_freq}
    # 每隔30秒,重新建立心跳连接:
    last_time = time.time() - start_time
    print("last_time:", last_time)
    if last_time % 30 == 0:  # 每隔30秒重新建立心跳
        print("重新建立ES连接")
        es_host = rag_test_Config().elastic_host
        es_port = 9200
        es = Elasticsearch(
            hosts=[{"host": es_host, "port": es_port, "scheme": "https"}],
            basic_auth=(rag_test_Config().elastic_username, rag_test_Config().elastic_password),
            verify_certs=False)
    es.index(index="finance_search_index", body=body)

last_time: 0.00035953521728515625
last_time: 0.15995526313781738
last_time: 0.16695928573608398
last_time: 0.1727449893951416
last_time: 0.1781482696533203
last_time: 0.18418073654174805
last_time: 0.19033074378967285
last_time: 0.19650769233703613
last_time: 0.2023477554321289
last_time: 0.20751285552978516
last_time: 0.21275067329406738
last_time: 0.2181227207183838
last_time: 0.22350788116455078
last_time: 0.2290024757385254
last_time: 0.23429560661315918
last_time: 0.23930954933166504
last_time: 0.2449781894683838
last_time: 0.2516794204711914
last_time: 0.2575724124908447
last_time: 0.264082670211792
last_time: 0.2701148986816406
last_time: 0.2793459892272949
last_time: 0.28603029251098633
last_time: 0.2930946350097656
last_time: 0.299670934677124
last_time: 0.3064703941345215
last_time: 0.313521146774292
last_time: 0.32027745246887207
last_time: 0.3264956474304199
last_time: 0.33307933807373047
last_time: 0.339508056640625
last_time: 0.3458681106567383
last_time: 0.35559916496276

In [38]:
count1 = es.count(index="finance_search_index")
count1

ObjectApiResponse({'count': 100, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

In [23]:
# 查询集合中所有数据
result=es.search(index="finance_search_index",body={"query":{"match_all":{}}})
for hit in result['hits']['hits']:
    print(hit['_source'])

{'finance_name': '中原证券交易中心', 'hot_freq': 57}
{'finance_name': '黄河金融交易所', 'hot_freq': 34}
{'finance_name': '郑州国际证券交易平台', 'hot_freq': 69}
{'finance_name': '豫州资本交易所', 'hot_freq': 85}
{'finance_name': '嵩山股权交易中心', 'hot_freq': 34}
{'finance_name': '洛阳龙门证券交易市场', 'hot_freq': 35}
{'finance_name': '商都金融交易所', 'hot_freq': 9}
{'finance_name': '郑东新区数字资产交易所', 'hot_freq': 7}
{'finance_name': '河南自贸区证券交易中心', 'hot_freq': 72}
{'finance_name': '安阳殷墟金融交易平台', 'hot_freq': 69}


In [25]:
query = {
    "query": {
        "match": {
            "finance_name": "郑州"  # 全文搜索，支持分词
        }
    }
}
response = es.search(index="finance_search_index", body=query)
for hit in response['hits']['hits']:
    print(hit['_source'])


{'finance_name': '郑州科技金融交易所', 'hot_freq': 46}
{'finance_name': '郑州国际证券交易平台', 'hot_freq': 69}
{'finance_name': '郑州商品证券交易中心', 'hot_freq': 50}
{'finance_name': '郑州智慧金融交易平台', 'hot_freq': 100}
{'finance_name': '郑州期货金融交易平台', 'hot_freq': 35}
{'finance_name': '郑州低碳资本交易市场', 'hot_freq': 26}
{'finance_name': '郑州数字资产交易市场', 'hot_freq': 19}
{'finance_name': '郑州电竞证券交易市场', 'hot_freq': 21}
{'finance_name': '郑州传媒金融交易市场', 'hot_freq': 9}
{'finance_name': '郑州航空港证券交易中心', 'hot_freq': 39}


In [28]:
# 根据query进行删除某个记录
delete_query = {
        "query": {
            "bool": {
                "must":[
                    {"match":{"finance_name":"郑州"}} # 线上启用状态
                ]
            }
        }
}
response = es.search(index="finance_search_index", body=delete_query, ignore=400)
response


ObjectApiResponse({'took': 2, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 21, 'relation': 'eq'}, 'max_score': 3.3756523, 'hits': [{'_index': 'finance_search_index', '_id': 'zehLPZ8BHFQ0I3JSixAA', '_score': 3.3756523, '_source': {'finance_name': '郑州科技金融交易所', 'hot_freq': 46}}, {'_index': 'finance_search_index', '_id': 'l-hLPZ8BHFQ0I3JSiRCn', '_score': 3.2340426, '_source': {'finance_name': '郑州国际证券交易平台', 'hot_freq': 69}}, {'_index': 'finance_search_index', '_id': 'tehLPZ8BHFQ0I3JSihBw', '_score': 3.2340426, '_source': {'finance_name': '郑州商品证券交易中心', 'hot_freq': 50}}, {'_index': 'finance_search_index', '_id': 'xehLPZ8BHFQ0I3JSihDT', '_score': 3.2340426, '_source': {'finance_name': '郑州智慧金融交易平台', 'hot_freq': 100}}, {'_index': 'finance_search_index', '_id': 'yehLPZ8BHFQ0I3JSihDq', '_score': 3.2340426, '_source': {'finance_name': '郑州期货金融交易平台', 'hot_freq': 35}}, {'_index': 'finance_search_index', '_id': '1ehLPZ8BHFQ0I3JSixA

In [31]:
response = es.delete_by_query(index="finance_search_index", body=delete_query, ignore=400)
print("Index delete response:", response)
#删除后结果
response = es.search(index="finance_search_index", body=delete_query, ignore=400)
for hit in response['hits']['hits']:
    print(hit['_source'])

Index delete response: {'took': 1, 'timed_out': False, 'total': 0, 'deleted': 0, 'batches': 0, 'version_conflicts': 0, 'noops': 0, 'retries': {'bulk': 0, 'search': 0}, 'throttled_millis': 0, 'requests_per_second': -1.0, 'throttled_until_millis': 0, 'failures': []}
